# 05. Text Leakage Investigation
**Author:** Member D  
**Project:** BuildSafe AI — NYC Construction Violation Severity Prediction  

## Objective
This notebook investigates target leakage and severity-adjacent language in `VIOLATION_DESCRIPTION` within `buildsafe_cleaned.csv`.

### Scope of Investigation
1. Load `data/processed/buildsafe_cleaned.csv`.
2. Search `VIOLATION_DESCRIPTION` for case-insensitive mentions of `"class"` (overall counts, percentages, and breakdown by target `SEVERITY`).
3. Inspect 25 un-truncated sample records containing `"class"` to analyze surrounding context.
4. Catalog every observed format/pattern of class number representations (e.g., `"CLASS 1"`, `"CLASS - 1"`, `"CLASS1"`, `"CLASS ONE"`) and distinguish genuine target leakage from domain-specific non-severity classes (e.g., Multiple Dwelling Law `"CLASS A"`, fire alarm `"CLASS J"`, and `"CLASSROOM"`).
5. Audit secondary severity-adjacent keywords (e.g., `"immediately hazardous"`, `"hazardous"`, `"major"`, `"lesser"`, `"imminent"`) and examine their distribution across target severity classes.

---

In [1]:
# Environment setup & imports
import pandas as pd
import numpy as np
import re

# Ensure full text displays without truncation
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Load Dataset

In [2]:
DATA_PATH = "../data/processed/buildsafe_cleaned.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Target SEVERITY distribution:")
print(df["SEVERITY"].value_counts(dropna=False))
print("\nTarget percentages:")
print((df["SEVERITY"].value_counts(normalize=True, dropna=False) * 100).round(2))

Loaded dataset: 102,467 rows × 22 columns
Target SEVERITY distribution:
SEVERITY
CLASS - 2    57607
CLASS - 1    40673
CLASS - 3     4187
Name: count, dtype: int64

Target percentages:
SEVERITY
CLASS - 2    56.22
CLASS - 1    39.69
CLASS - 3     4.09
Name: proportion, dtype: float64


## 2. Search `VIOLATION_DESCRIPTION` for Mentions of "class"

We perform a case-insensitive search across `VIOLATION_DESCRIPTION` for:
1. Any case-insensitive substring mention of `"class"` (to capture variations like `"CLASS1"`).
2. Word boundary `r"\bclass\b"` mentions.

In [3]:
desc = df["VIOLATION_DESCRIPTION"].fillna("").astype(str)

# Substring match (case-insensitive)
df["has_class_substring"] = desc.str.contains("class", case=False, regex=False)

# Word boundary match (case-insensitive)
df["has_class_word"] = desc.str.contains(r"\bclass\b", case=False, regex=True)

total_rows = len(df)
sub_count = int(df["has_class_substring"].sum())
sub_pct = (sub_count / total_rows) * 100

word_count = int(df["has_class_word"].sum())
word_pct = (word_count / total_rows) * 100

print(f"Total rows in dataset: {total_rows:,}")
print(f"Rows containing 'class' (case-insensitive substring): {sub_count:,} ({sub_pct:.2f}%)")
print(f"Rows containing 'class' (word boundary \\bclass\\b):    {word_count:,} ({word_pct:.2f}%)")

Total rows in dataset: 102,467
Rows containing 'class' (case-insensitive substring): 7,378 (7.20%)
Rows containing 'class' (word boundary \bclass\b):    6,576 (6.42%)


### Breakdown by Target `SEVERITY`

In [4]:
# Breakdown for substring match
breakdown_sub = pd.crosstab(
    df["SEVERITY"],
    df["has_class_substring"],
    margins=True
).rename(columns={False: "No 'class'", True: "Contains 'class'", "All": "Total Rows"})

breakdown_sub["Pct_Within_Severity_Class (%)"] = (
    breakdown_sub["Contains 'class'"] / breakdown_sub["Total Rows"] * 100
).round(2)

breakdown_sub["Share_Of_All_Mentions (%)"] = (
    breakdown_sub["Contains 'class'"] / sub_count * 100
).round(2)

print("=== BREAKDOWN BY SEVERITY (ANY 'CLASS' MENTION) ===")
print(breakdown_sub.to_string())

=== BREAKDOWN BY SEVERITY (ANY 'CLASS' MENTION) ===
has_class_substring  No 'class'  Contains 'class'  Total Rows  Pct_Within_Severity_Class (%)  Share_Of_All_Mentions (%)
SEVERITY                                                                                                               
CLASS - 1                 37069              3604       40673                           8.86                      48.85
CLASS - 2                 53849              3758       57607                           6.52                      50.94
CLASS - 3                  4171                16        4187                           0.38                       0.22
All                       95089              7378      102467                           7.20                     100.00


## 3. 25 Random Un-truncated Example Rows Matching "class"

We inspect 25 random examples with full un-truncated `VIOLATION_DESCRIPTION` text to evaluate the surrounding operational and legal context.

In [5]:
matched_rows = df[df["has_class_substring"]].copy()

# Sample 25 rows with fixed random_state for reproducible inspection
sample_25 = matched_rows.sample(n=25, random_state=42)

for i, (idx, row) in enumerate(sample_25.iterrows(), 1):
    print("=" * 90)
    print(f"SAMPLE {i:02d} | Row Index: {idx} | Actual Target SEVERITY: {row['SEVERITY']}")
    print("-" * 90)
    print(row["VIOLATION_DESCRIPTION"])
    print()


SAMPLE 01 | Row Index: 17142 | Actual Target SEVERITY: CLASS - 2
------------------------------------------------------------------------------------------
CLASS 2 ITEMS: IN-CAR COMMUNICATION INOPERATIVE. PROVIDE CODE DATA TAGWITH ALL INFORMATION.BRAKE MAINTENANCE TAG EXPIRED. CATEGORY 5 TEST TAG MISSING. PROVIDE.

SAMPLE 02 | Row Index: 34436 | Actual Target SEVERITY: CLASS - 1
------------------------------------------------------------------------------------------
WORK WITHOUT PERMIT. AT TIME OF INSPECTION, OBSERVED FULL HEIGHT PARTITION WALLS ERECTED TO SUB-DIVIDE THE CELLAR CREATING ILLEGAL CLASS A APARTMENT WITH S.R.O ROOM, GAS STOVE/RESIDENTIAL SINK KITCHEN, AND 3 PC

SAMPLE 03 | Row Index: 100142 | Actual Target SEVERITY: CLASS - 1
------------------------------------------------------------------------------------------
OCCUPANCY CONTRARY TO THAT ALLOWED BY THE CERTIFICATE OF OCCUPANCY OR BUILDING DEPT. RECORDS. AT TIME OF INSPECTION OBSERVED BASEMENT LEVEL TURNED INTO CLASS 

## 4. Distinct Formats and Patterns Observed

From our analysis of the matched records and surrounding context, the appearance of the word `"class"` divides strictly into **two categories**:

### A. Direct Violation Severity Mentions (True Target Leakage)
In thousands of records, the inspector explicitly notes the violation severity class in the narrative. The following distinct formatting patterns appear in the text:

| Pattern Format | Example from Data | Context / Meaning |
|---|---|---|
| `CLASS [1-3] ITEMS:` / `CLASS [1-3] ITEM:` | `CLASS 1 ITEMS: 1) NO DOOR LOCK MONITORING...` | Inspector categorizing infractions by severity |
| `CLASS [1-3]:` (with colon) | `CLASS 1: CEASE USE: DOOR RESTRICTOR INOPERATIVE...` | Direct severity prefix |
| `CLASS [1-3]` (space separated) | `FOR CLASS 1 WORK WITH OUT PERMIT...`, `NO CLASS 1 CONDITIONS OBSERVED` | Descriptive classification of work or condition |
| `CLASS - [1-3]` (spaced hyphen) | `CLASS - 1`, `CLASS - 2` | Same format as target label |
| `CLASS-[1-3]` (direct hyphen) | `CLASS-1`, `CLASS-2` | Hyphenated abbreviation |
| `CLASS[1-3]` (no space) | `CLASS1`, `CLASS2` | Concatenated format |
| `CLASS #[1-3]` (with hash) | `CLASS #1`, `CLASS #2` | Numbered notation |
| `CLASS [ONE|TWO|THREE]` | `CLASS ONE`, `CLASS TWO` | Spelled-out number |
| `CLASS [I|II|III]` | `CLASS I`, `CLASS II` | Roman numeral notation |
| `CLASS 1 RTS` / `CLASS 2 RTS` | `CLASS 1 RTS:FAILURE TO MAINTAIN...` | DOB abbreviation: "Restore To Service" |
| Multiple classes in single record | `CLASS 1: CEASE USE... CLASS 2: CAB POSITION...` | Compound violation containing items of differing severity |

### B. Domain-Specific Non-Severity Uses of "Class" (Legitimate Building Context)
Crucially, not every mention of `"class"` is target leakage. Several hundred records use `"class"` for unrelated statutory or architectural designations:

1. **Multiple Dwelling Law (MDL) Occupancy Classification:**
   - Formats: `CLASS A`, `CLASS 'A'`, `CLASS "A"`, `CLASS (A)`, `CLASS-A`, `CLASS"A"`, `CLASS B`.
   - Context: Refers to legal building occupancy classes under NYC Multiple Dwelling Law (e.g., `"ILLEGAL CLASS A APARTMENT CREATED AT CELLAR"`, `"CONVERTED TO TRANSIENT HAVI CLASS A MULTIPLE DWELLING"`). This describes illegal residential conversions, not violation severity.
2. **Fire Alarm System Specifications:**
   - Formats: `CLASS 'J' FIRE ALARM SYSTEM`, `CLASS "J"`, `CLASS J`.
   - Context: Refers to NYC Building Code Class J fire alarm systems for high-rise or transient occupancies.
3. **Architectural / School Room Types:**
   - Formats: `CLASSROOM`, `CLASSROOMS`, `CLASS ROOM`, `CLASS ROOMS`.
   - Context: Describes physical rooms (e.g., `"CLASSROOM DOOR MISS SELF-CLOSING CHECKS"`).
4. **General Vocabulary:**
   - Words like `CLASSIFIED`, `CLASSIFICATION`.

In [6]:
# Quantitative breakdown: Severity-class leakage vs. Occupancy / Classroom mentions
pat_severity = r'(?i)\bclass\s*[-:#]?\s*(?:[123]|one|two|three|i{1,3})\b|\bclass[123]\b'
pat_occupancy = r'(?i)\bclass\s*[-_"\'\(\[]?\s*[abj]\b'
pat_classroom = r'(?i)\bclass\s*rooms?\b|\bclassrooms?\b'

df["mentions_severity_class"] = desc.str.contains(pat_severity, regex=True)
df["mentions_occupancy_class"] = desc.str.contains(pat_occupancy, regex=True)
df["mentions_classroom"] = desc.str.contains(pat_classroom, regex=True)

print(f"Mentions direct severity class (1, 2, 3, etc.): {df['mentions_severity_class'].sum():,} ({df['mentions_severity_class'].mean()*100:.2f}%)")
print(f"Mentions occupancy class (Class A, B, J):        {df['mentions_occupancy_class'].sum():,} ({df['mentions_occupancy_class'].mean()*100:.2f}%)")
print(f"Mentions classroom / rooms:                     {df['mentions_classroom'].sum():,} ({df['mentions_classroom'].mean()*100:.2f}%)")

print("\nCross-tabulation of Direct Severity Class Mentions vs. Actual SEVERITY:")
ct_sev = pd.crosstab(df["SEVERITY"], df["mentions_severity_class"], margins=True)
ct_sev["% with Severity Class"] = (ct_sev[True] / ct_sev["All"] * 100).round(2)
print(ct_sev)


Mentions direct severity class (1, 2, 3, etc.): 4,273 (4.17%)
Mentions occupancy class (Class A, B, J):        2,282 (2.23%)
Mentions classroom / rooms:                     659 (0.64%)

Cross-tabulation of Direct Severity Class Mentions vs. Actual SEVERITY:
mentions_severity_class  False  True     All  % with Severity Class
SEVERITY                                                           
CLASS - 1                38578  2095   40673                   5.15
CLASS - 2                55430  2177   57607                   3.78
CLASS - 3                 4186     1    4187                   0.02
All                      98194  4273  102467                   4.17


## 5. Investigation of Severity-Adjacent Words

Under NYC Administrative Code § 28-201.2, violations are categorized into three statutory degrees of severity:
- **CLASS - 1:** *Immediately Hazardous*
- **CLASS - 2:** *Major*
- **CLASS - 3:** *Lesser*

We test whether descriptions contain these statutory severity terms or related severity-adjacent language, how frequently they occur, and whether they disproportionately concentrate in specific classes.

In [7]:
severity_keywords = [
    "immediately hazardous",
    "hazardous",
    "hazard",
    "major",
    "lesser",
    "imminent",
    "non-hazardous",
    "severity",
    "aggravated",
    "emergency",
    "life safety",
    "high risk"
]

desc_lower = desc.str.lower()
keyword_records = []

for kw in severity_keywords:
    pattern = r"\b" + kw + r"\b"
    match_series = desc_lower.str.contains(pattern, regex=True)
    count = int(match_series.sum())
    pct = (count / total_rows) * 100
    
    c1 = int((match_series & (df["SEVERITY"] == "CLASS - 1")).sum())
    c2 = int((match_series & (df["SEVERITY"] == "CLASS - 2")).sum())
    c3 = int((match_series & (df["SEVERITY"] == "CLASS - 3")).sum())
    
    keyword_records.append({
        "Keyword": kw,
        "Total Matches": count,
        "Dataset Pct (%)": round(pct, 3),
        "CLASS - 1": c1,
        "CLASS - 2": c2,
        "CLASS - 3": c3,
        "% Class 1": round(c1 / count * 100, 1) if count > 0 else 0.0,
        "% Class 2": round(c2 / count * 100, 1) if count > 0 else 0.0,
        "% Class 3": round(c3 / count * 100, 1) if count > 0 else 0.0
    })

kw_summary_df = pd.DataFrame(keyword_records)
print("=== AUDIT OF SEVERITY-ADJACENT KEYWORDS IN VIOLATION_DESCRIPTION ===")
print(kw_summary_df.to_string(index=False))

=== AUDIT OF SEVERITY-ADJACENT KEYWORDS IN VIOLATION_DESCRIPTION ===
              Keyword  Total Matches  Dataset Pct (%)  CLASS - 1  CLASS - 2  CLASS - 3  % Class 1  % Class 2  % Class 3
immediately hazardous             18            0.018         13          5          0       72.2       27.8        0.0
            hazardous           2582            2.520       1460       1112         10       56.5       43.1        0.4
               hazard           2740            2.674       1373       1287         80       50.1       47.0        2.9
                major            180            0.176        122         57          1       67.8       31.7        0.6
               lesser              0            0.000          0          0          0        0.0        0.0        0.0
             imminent             23            0.022         21          2          0       91.3        8.7        0.0
        non-hazardous              0            0.000          0          0          0     

### Key Observations on Severity-Adjacent Terms:
1. **`"immediately hazardous"`:** Appears 18 times; 72.2% of occurrences are `CLASS - 1` (the legal statutory name for Class 1).
2. **`"hazardous"` & `"hazard"`:** Very frequent (2,582 and 2,740 rows respectively, ~2.5-2.7% of the dataset). They are split roughly 56% / 43% across Class 1 and Class 2, but virtually absent from Class 3 (<0.5%).
3. **`"major"`:** Appears in 180 rows, but interestingly 67.8% are Class 1 rather than Class 2 (often describing "major alteration without a permit").
4. **`"lesser"` & `"non-hazardous"`:** Appear **0** times in the entire corpus. Inspectors never use the statutory term "lesser" in narrative text.
5. **`"emergency"`:** Appears in 2,197 rows, with 75.6% falling into `CLASS - 2` (frequently elevator/lighting emergency systems).
6. **`"imminent"` & `"life safety"`:** Rare (23 and 27 rows), but strongly skewed toward `CLASS - 1` (91.3% and 81.5%).

## 6. Summary & Recommendations for Member D

### Findings
1. **`"class"` Mentions:** 7,378 rows (7.20%) contain the substring `"class"`. Of these:
   - **4,273 rows (4.17%)** contain direct severity-class mentions (`CLASS 1`, `CLASS 2`, `CLASS 3`, `CLASS ONE`, etc.).
   - **2,282 rows (2.23%)** contain legal occupancy designations (`CLASS A`, `CLASS B`, `CLASS J`).
   - **659 rows (0.64%)** refer to physical school/facility spaces (`CLASSROOM`).
2. **Target Leakage Impact:** Direct severity mentions provide substantial target leakage to NLP models (e.g. TF-IDF unigrams/bigrams like `class`, `class 1`, `class 2`), explaining why the baseline Linear SVM in Notebook 04 achieved an artificially high ~99.4% accuracy.
3. **Surgical Scrubbing vs. Blanket Removal:** A naive removal of the token `"class"` would strip valuable predictive signal from Multiple Dwelling Law violations (`"CLASS A APARTMENT"`) and facility descriptions (`"CLASSROOM"`). Scrubbing must specifically target regex patterns of severity levels (`CLASS [1-3]`, `CLASS ONE/TWO/THREE`, `CLASS-1`, `CLASS1`, etc.).

## 7. Scrubbing Target Severity Mentions from `VIOLATION_DESCRIPTION`

We apply the validated regex `pat_severity` from Section 4 to excise direct severity labels from `VIOLATION_DESCRIPTION` into a new column `VIOLATION_DESCRIPTION_CLEAN`. Matches are replaced with a single space, and any resulting multiple spaces are collapsed. Occupancy classifications (Class A/B/J) and classroom references are preserved.

In [8]:
# ============================================================
# TASK 1: CREATE VIOLATION_DESCRIPTION_CLEAN
# ============================================================

def clean_description(text):
    if pd.isna(text):
        return ""
    # Replace severity class matches with a single space
    cleaned = re.sub(pat_severity, " ", str(text))
    # Collapse double spaces and strip whitespace
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

df["VIOLATION_DESCRIPTION_CLEAN"] = df["VIOLATION_DESCRIPTION"].apply(clean_description)

print("Created VIOLATION_DESCRIPTION_CLEAN column successfully.")
print(f"Total rows: {len(df):,}")


Created VIOLATION_DESCRIPTION_CLEAN column successfully.
Total rows: 102,467


## 8. Verification of Scrubbing Fix

We re-run the severity-class and substring checks on `VIOLATION_DESCRIPTION_CLEAN` to verify that severity matches dropped to zero, and inspect any remaining class mentions.

In [9]:
# ============================================================
# TASK 2: VERIFICATION OF THE FIX
# ============================================================

clean_desc = df["VIOLATION_DESCRIPTION_CLEAN"].fillna("").astype(str)

# Check remaining severity-class matches
rem_severity_matches = clean_desc.str.contains(pat_severity, regex=True)
rem_sev_count = int(rem_severity_matches.sum())

# Check remaining 'class' substring matches
rem_class_substring = clean_desc.str.contains("class", case=False, regex=False)
rem_sub_count = int(rem_class_substring.sum())

# Check remaining occupancy class matches
rem_occupancy = clean_desc.str.contains(pat_occupancy, regex=True)
rem_occ_count = int(rem_occupancy.sum())

# Check remaining classroom matches
rem_classroom = clean_desc.str.contains(pat_classroom, regex=True)
rem_room_count = int(rem_classroom.sum())

print(f"Original severity class matches:   {df['mentions_severity_class'].sum():,}")
print(f"Remaining severity class matches:  {rem_sev_count:,} (Expected: 0)")
print(f"Remaining 'class' substring count: {rem_sub_count:,}")
print(f"  - Preserved Occupancy (A/B/J):   {rem_occ_count:,}")
print(f"  - Preserved Classrooms:          {rem_room_count:,}")

# Print any remaining severity matches in full if any exist
if rem_sev_count > 0:
    print(f"\nRemaining {rem_sev_count} severity matches found (review for edge cases):")
    for idx, row in df[rem_severity_matches].iterrows():
        print(f"[Row {idx}] {row['VIOLATION_DESCRIPTION_CLEAN']}")
else:
    print("\nVerified: Zero severity class mentions remain in VIOLATION_DESCRIPTION_CLEAN!")


Original severity class matches:   4,273
Remaining severity class matches:  0 (Expected: 0)
Remaining 'class' substring count: 3,153
  - Preserved Occupancy (A/B/J):   2,282
  - Preserved Classrooms:          659

Verified: Zero severity class mentions remain in VIOLATION_DESCRIPTION_CLEAN!


## 9. Before vs. After Comparison (Side-by-Side)

We inspect 10 representative before and after examples, covering both records where severity was stripped and records where occupancy/classroom mentions were preserved.

In [10]:
# ============================================================
# TASK 3: 10 BEFORE / AFTER EXAMPLES SIDE-BY-SIDE
# ============================================================

# Select a balanced set of examples:
# 6 where severity was removed, 2 occupancy preserved, 2 classroom preserved
severity_indices = df[df["mentions_severity_class"]].index[:6].tolist()
occupancy_indices = df[df["mentions_occupancy_class"] & ~df["mentions_severity_class"]].index[:2].tolist()
classroom_indices = df[df["mentions_classroom"] & ~df["mentions_severity_class"]].index[:2].tolist()

selected_indices = severity_indices + occupancy_indices + classroom_indices

for i, idx in enumerate(selected_indices, 1):
    category = "Severity Scrubbed" if idx in severity_indices else ("Occupancy Preserved" if idx in occupancy_indices else "Classroom Preserved")
    print("=" * 95)
    print(f"EXAMPLE {i:02d} [{category}] | Row Index: {idx} | Target SEVERITY: {df.loc[idx, 'SEVERITY']}")
    print("-" * 95)
    print("ORIGINAL:")
    print(df.loc[idx, "VIOLATION_DESCRIPTION"])
    print("\nCLEANED:")
    print(df.loc[idx, "VIOLATION_DESCRIPTION_CLEAN"])
    print()


EXAMPLE 01 [Severity Scrubbed] | Row Index: 21 | Target SEVERITY: CLASS - 2
-----------------------------------------------------------------------------------------------
ORIGINAL:
CLASS 2: PHONE INOPERATIVE REAPIR & CONNECT SERVICE TO 24 HR EMERGENCY OPERATOR. CAR DOOR MISALIGNED CAUSING RUBBING/SCRAPPING ADJUST TO REMOVE. NO ACCESS TO MOTOR ROOM PROVIDE KEYS & ACCESS FOR INSPECTION. CLE

CLEANED:
: PHONE INOPERATIVE REAPIR & CONNECT SERVICE TO 24 HR EMERGENCY OPERATOR. CAR DOOR MISALIGNED CAUSING RUBBING/SCRAPPING ADJUST TO REMOVE. NO ACCESS TO MOTOR ROOM PROVIDE KEYS & ACCESS FOR INSPECTION. CLE

EXAMPLE 02 [Severity Scrubbed] | Row Index: 116 | Target SEVERITY: CLASS - 2
-----------------------------------------------------------------------------------------------
ORIGINAL:
CLASS 2: PHONE INOPERATIVE REPAIR & CONNECT SERVICE TO 24 HR EMERGENCY OPERATOR. SEASON HOIST ROPES & ADJUST CLIPS TO CODE. PROVIDE LEGIBLE BRAKE MAINTANENCE TAG. CLEAN GARBAGE FROM PIT.

CLEANED:
: PHONE INOP

## 10. Save Text-Cleaned Dataset

We save the resulting dataset with the new `VIOLATION_DESCRIPTION_CLEAN` column to `data/processed/buildsafe_text_cleaned.csv` without modifying `buildsafe_cleaned.csv`.

In [11]:
# ============================================================
# TASK 4: SAVE TO BUILDSAFE_TEXT_CLEANED.CSV
# ============================================================

OUTPUT_CLEAN_PATH = "../data/processed/buildsafe_text_cleaned.csv"

df.to_csv(OUTPUT_CLEAN_PATH, index=False)

print("Saved cleaned text dataset successfully.")
print("File path:", OUTPUT_CLEAN_PATH)
print("Dataset shape:", df.shape)
print("New columns:", [c for c in df.columns if "CLEAN" in c])


Saved cleaned text dataset successfully.
File path: ../data/processed/buildsafe_text_cleaned.csv
Dataset shape: (102467, 28)
New columns: ['VIOLATION_DESCRIPTION_CLEAN']


## 11. Sanity Check Model (Throwaway Benchmark Only)

> **NOTE:** This is a throwaway sanity check, NOT the real final model or final TF-IDF pipeline.

We fit a quick TF-IDF (`max_features=2000`, `stop_words="english"`) and a basic `LogisticRegression` on an 80/20 train/test split, once using `VIOLATION_DESCRIPTION` (original) and once using `VIOLATION_DESCRIPTION_CLEAN` (scrubbed), to observe the change in accuracy and macro-F1.

In [12]:
# ============================================================
# TASK 5: SANITY CHECK (THROWAWAY BENCHMARK ONLY)
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Simple train/test split on text
X_orig = df["VIOLATION_DESCRIPTION"].fillna("")
X_clean = df["VIOLATION_DESCRIPTION_CLEAN"].fillna("")
y = df["SEVERITY"]

X_tr_orig, X_te_orig, y_train, y_test = train_test_split(
    X_orig, y, test_size=0.20, random_state=42, stratify=y
)

X_tr_clean, X_te_clean, _, _ = train_test_split(
    X_clean, y, test_size=0.20, random_state=42, stratify=y
)

# 1. Benchmark on ORIGINAL text
tfidf_orig = TfidfVectorizer(max_features=2000, stop_words="english")
X_tr_orig_vec = tfidf_orig.fit_transform(X_tr_orig)
X_te_orig_vec = tfidf_orig.transform(X_te_orig)

clf_orig = LogisticRegression(max_iter=1000, random_state=42)
clf_orig.fit(X_tr_orig_vec, y_train)
y_pred_orig = clf_orig.predict(X_te_orig_vec)

acc_orig = accuracy_score(y_test, y_pred_orig)
f1_orig = f1_score(y_test, y_pred_orig, average="macro")

# 2. Benchmark on CLEANED text
tfidf_clean = TfidfVectorizer(max_features=2000, stop_words="english")
X_tr_clean_vec = tfidf_clean.fit_transform(X_tr_clean)
X_te_clean_vec = tfidf_clean.transform(X_te_clean)

clf_clean = LogisticRegression(max_iter=1000, random_state=42)
clf_clean.fit(X_tr_clean_vec, y_train)
y_pred_clean = clf_clean.predict(X_te_clean_vec)

acc_clean = accuracy_score(y_test, y_pred_clean)
f1_clean = f1_score(y_test, y_pred_clean, average="macro")

print("=" * 65)
print("SANITY CHECK RESULTS (THROWAWAY BENCHMARK ONLY)")
print("=" * 65)
print(f"Original Text Model -> Accuracy: {acc_orig*100:.2f}%, Macro-F1: {f1_orig*100:.2f}%")
print(f"Cleaned Text Model  -> Accuracy: {acc_clean*100:.2f}%, Macro-F1: {f1_clean*100:.2f}%")
print(f"Difference (Delta)  -> Accuracy: {(acc_clean - acc_orig)*100:+.2f}%, Macro-F1: {(f1_clean - f1_orig)*100:+.2f}%")
print("=" * 65)

print("\n--- Cleaned Text Classification Report ---")
print(classification_report(y_test, y_pred_clean, digits=4))


SANITY CHECK RESULTS (THROWAWAY BENCHMARK ONLY)
Original Text Model -> Accuracy: 79.50%, Macro-F1: 72.81%
Cleaned Text Model  -> Accuracy: 79.59%, Macro-F1: 73.00%
Difference (Delta)  -> Accuracy: +0.09%, Macro-F1: +0.19%

--- Cleaned Text Classification Report ---


              precision    recall  f1-score   support

   CLASS - 1     0.7732    0.7774    0.7753      8135
   CLASS - 2     0.8144    0.8310    0.8226     11522
   CLASS - 3     0.7401    0.4934    0.5921       837

    accuracy                         0.7959     20494
   macro avg     0.7759    0.7006    0.7300     20494
weighted avg     0.7950    0.7959    0.7944     20494

